# Aeolus — Demo

This notebook serves as an interactive test client for the `aeolus` library, before a proper user interface is built.

## How it works

1. **Upload a GPX file** – the route is split into clusters based on speed and gradient
2. **Fetch weather data** – a 15-minute forecast is retrieved from [Open-Meteo](https://open-meteo.com/) for each cluster's midpoint
3. **Calculate score** – each segment is scored by wind direction and strength, gustiness, and precipitation, aggregated by distance
4. **Show map** – wind arrows indicate direction and speed; the route is colored by segment score (green → red), dashed where conditions are flagged unsafe

## Score Scale

| Score | Meaning |
|-------|---------|
| **0.5 – 1.0** | Good conditions – tailwind, no gusts, dry |
| **0.0 – 0.5** | Acceptable conditions – light crosswind or headwind |
| **−0.5 – 0.0** | Difficult conditions – stronger headwind or rain |
| **−1.0 – −0.5** | Bad conditions – strong gusts or heavy rain |

The score is dimensionless and distance-weighted: short segments with poor weather pull the overall score down less than long ones.

Whether conditions are dangerous is reported separately from the score, so a storm with a tailwind and a storm with a headwind stay distinguishable. The defaults behind these numbers are informed guesses that have not yet been calibrated against recorded rides.

## Interactive Analysis

Upload a GPX file, set your average speed and start time, then click **Analyze Route**.

In [ ]:
from datetime import datetime
import ipywidgets as widgets
from IPython.display import display
import base64
import folium
from app.services.gpx_parser import get_clustered_route, parse_gpx
from app.services.weather import get_weather_for_route
from app.services.route_scorer import score_segment, DEFAULT_PARAMS

file_upload = widgets.FileUpload(
    accept=".gpx",
    multiple=False,
    description="GPX File",
    style={"button_color": "lightblue"},
)

speed_slider = widgets.FloatSlider(
    value=15.0,
    min=1.0,
    max=60.0,
    step=0.5,
    description="Avg Speed (km/h):",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="450px"),
)

start_time_picker = widgets.NaiveDatetimePicker(
    value=datetime.now().replace(second=0, microsecond=0),
    description="Start Time:",
    style={"description_width": "initial"},
)

submit_button = widgets.Button(
    description="Analyze Route",
    button_style="primary",
    icon="play",
    layout=widgets.Layout(margin="8px 0"),
)

score_output = widgets.Output()
map_widget = widgets.HTML(value="", layout=widgets.Layout(width="100%", height="500px"))


def _score_to_color(score: float) -> str:
    """Interpolates from red (−1) through yellow (0) to green (1)."""
    t = (score + 1.0) / 2.0  # 0..1
    if t < 0.5:
        r, g = 220, int(t * 2 * 200)
    else:
        r, g = int((1 - t) * 2 * 220), 180
    return f"#{r:02x}{g:02x}32"


def _score_label(score: float) -> str:
    """Returns a plain-language label for a given score."""
    if score >= 0.5:
        return "Good conditions"
    elif score >= 0.0:
        return "Acceptable conditions"
    elif score >= -0.5:
        return "Difficult conditions"
    else:
        return "Bad conditions"


def _weighted_breakdown(snapshots, scores, total_distance_m):
    """Distance-weighted averages of the score components and their inputs."""
    totals = dict.fromkeys(
        ["wind", "gust", "rain", "speed", "gusts", "precip", "unsafe"], 0.0
    )
    alignments: dict[str, float] = {}

    for snap, score in zip(snapshots, scores):
        weight = snap.cluster.total_distance_m / total_distance_m

        totals["wind"] += score.wind * weight
        totals["gust"] += score.gust * weight
        totals["rain"] += score.rain * weight
        totals["speed"] += snap.wind_speed_km_h * weight
        totals["gusts"] += snap.wind_gusts_km_h * weight
        totals["precip"] += score.precipitation_mm_h * weight
        totals["unsafe"] += weight if score.unsafe else 0.0

        label = score.alignment.value
        alignments[label] = alignments.get(label, 0.0) + weight

    totals["dominant_wind"] = max(alignments, key=alignments.get)
    return totals


def plot_route(snapshots, gpx_bytes, scores):
    points = parse_gpx(gpx_bytes)
    lats = [p.lat for p in points]
    lons = [p.lon for p in points]
    center = [sum(lats) / len(lats), sum(lons) / len(lons)]

    m = folium.Map(location=center, zoom_start=11)

    # Color route segments by score, dashed where flagged unsafe
    for snap, score in zip(snapshots, scores):
        segs = snap.cluster.segments
        coords = [(segs[0].start.lat, segs[0].start.lon)] + [
            (s.end.lat, s.end.lon) for s in segs
        ]
        folium.PolyLine(
            coords,
            color=_score_to_color(score.score),
            weight=5,
            opacity=0.85,
            dash_array="8" if score.unsafe else None,
            tooltip=f"Score: {score.score:+.2f} ({score.alignment.value})",
        ).add_to(m)

    # Wind arrows
    for snap, score in zip(snapshots, scores):
        rp = snap.cluster.representative_point
        arrow_deg = (snap.wind_direction_deg + 180) % 360

        if score.precipitation_mm_h >= 4.0:
            arrow_color = "#0000cc"
        elif score.precipitation_mm_h >= 0.4:
            arrow_color = "#4444ff"
        else:
            arrow_color = "#888888"

        icon_html = f"""
        <div style="width:44px; text-align:center; line-height:1.1;">
          <div style="transform:rotate({arrow_deg:.0f}deg); font-size:20px; color:{arrow_color};">&#8593;</div>
          <div style="font-size:9px; color:#222;">{snap.wind_speed_km_h:.0f}&nbsp;km/h</div>
        </div>
        """
        popup_html = (
            f"<b>Time:</b> {snap.timestamp.strftime('%H:%M')}<br>"
            f"<b>Score:</b> {score.score:+.2f}<br>"
            f"<b>Wind:</b> {snap.wind_speed_km_h:.1f} km/h "
            f"({score.alignment.value}, {score.wind:+.2f})<br>"
            f"<b>Gusts:</b> {snap.wind_gusts_km_h:.1f} km/h ({score.gust:+.2f})<br>"
            f"<b>Rain:</b> {score.precipitation_mm_h:.2f} mm/h ({score.rain:+.2f})"
            + ("<br><b>⚠ Unsafe conditions</b>" if score.unsafe else "")
        )
        folium.Marker(
            location=[rp.lat, rp.lon],
            popup=folium.Popup(popup_html, max_width=240),
            icon=folium.DivIcon(html=icon_html, icon_size=(44, 44), icon_anchor=(22, 22)),
        ).add_to(m)

    map_html = m.get_root().render()
    encoded = base64.b64encode(map_html.encode()).decode()
    map_widget.value = f'<iframe src="data:text/html;base64,{encoded}" width="100%" height="500px" frameborder="0"></iframe>'


def on_submit(b):
    score_output.clear_output()
    map_widget.value = ""
    submit_button.disabled = True
    submit_button.description = "Analyzing..."

    try:
        with score_output:
            if not file_upload.value:
                print("Please upload a GPX file first.")
                return

            content = file_upload.value[0]["content"].tobytes()
            print("Segmenting route...")
            route_clusters = get_clustered_route(content, speed_slider.value, start_time_picker.value)
            n = len(route_clusters.clusters)
            dist_km = route_clusters.total_distance_m / 1000
            print(f"  → {n} segments, {dist_km:.1f} km total distance, starting at {start_time_picker.value.strftime('%Y-%m-%d %H:%M')} estimated arrival at {route_clusters.representative_points[-1].timestamp.strftime('%Y-%m-%d %H:%M')}")

            print("Fetching weather data...")
            snapshots = get_weather_for_route(route_clusters)
            print(f"  → {len(snapshots)} forecasts loaded")

            print("Calculating score...")
            scores = [score_segment(s) for s in snapshots]
            total_score = sum(
                score.score * snap.cluster.total_distance_m
                for score, snap in zip(scores, snapshots)
            ) / route_clusters.total_distance_m

            breakdown = _weighted_breakdown(snapshots, scores, route_clusters.total_distance_m)
            params = DEFAULT_PARAMS
            rule = "─" * 46

            print(
                f"\n{rule}\n"
                f"  Overall Score: {total_score:+.2f}  |  {_score_label(total_score)}\n"
                f"{rule}\n"
                f"  Why:\n"
                f"  Wind  (x{params.wind_weight:.2f}): {breakdown['wind'] * params.wind_weight:+.2f}"
                f"  →  avg {breakdown['speed']:.1f} km/h, mostly {breakdown['dominant_wind']}\n"
                f"  Gusts (x{params.gust_weight:.2f}): {breakdown['gust'] * params.gust_weight:+.2f}"
                f"  →  avg {breakdown['gusts']:.1f} km/h\n"
                f"  Rain  (x{params.rain_weight:.2f}): {breakdown['rain'] * params.rain_weight:+.2f}"
                f"  →  avg {breakdown['precip']:.2f} mm/h\n"
                f"{rule}"
            )
            if breakdown["unsafe"] > 0.0:
                print(
                    f"  ⚠ {breakdown['unsafe'] * 100:.0f} % of the route exceeds a safety\n"
                    f"    threshold — shown dashed on the map\n"
                    f"{rule}"
                )

        plot_route(snapshots, content, scores)

    except Exception as e:
        with score_output:
            print(f"\nError: {e}")
    finally:
        submit_button.disabled = False
        submit_button.description = "Analyze Route"


submit_button._click_handlers.callbacks.clear()
submit_button.on_click(on_submit)


In [ ]:
display(
    file_upload,
    speed_slider,
    start_time_picker,
    submit_button,
    score_output,
    map_widget,
)
